# Customer Churn: EDA + Preprocessing + Model
Simple step-by-step. Just run each cell.

## Step 1: Load data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("classification_customer_churn.csv")
df.head()

## Step 2: EDA (look at the data)

In [ ]:
print(df.shape)
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing values:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())

**Target balance** (how many leave vs stay)

In [ ]:
print(df["churn"].value_counts(normalize=True))
sns.countplot(data=df, x="churn")
plt.show()

**Numeric columns**

In [ ]:
num_cols = ["tenure_months", "monthly_charges", "support_calls", "age"]
df[num_cols].hist(bins=20, figsize=(10, 6))
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, col in zip(ax, num_cols):
    sns.boxplot(data=df, x="churn", y=col, ax=a)
plt.show()

**Categorical columns vs churn** (average churn rate per group)

In [ ]:
for col in ["contract_type", "internet_service", "paperless_billing"]:
    print(df.groupby(col)["churn"].mean().round(2), "\n")

**Correlation**

In [ ]:
sns.heatmap(df.select_dtypes("number").corr(), annot=True, cmap="coolwarm")
plt.show()

## Step 3: Preprocessing

Split first, so the test data stays unseen.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="churn")
y = df["churn"]
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(x_train.shape, x_test.shape)

Turn text columns into numbers (one-hot) and scale numeric columns.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num = ["tenure_months", "monthly_charges", "support_calls", "age", "paperless_billing"]
cat = ["contract_type", "internet_service"]

pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(drop="first"), cat),
])

## Step 4: Train models and compare

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(learning_rate=0.05, max_depth=4, random_state=42),
}

log = []
for name, m in models.items():
    pipe = Pipeline([("pre", pre), ("clf", m)])
    pipe.fit(x_train, y_train)
    pred = pipe.predict(x_test)
    log.append({"model": name,
                "accuracy": accuracy_score(y_test, pred),
                "f1": f1_score(y_test, pred)})   # binary, so no average needed

results = pd.DataFrame(log).round(3)
results.to_csv("experiments.csv", index=False)
results

## Step 5: Detailed report for the best model

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

best = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(x_train, y_train)
print(classification_report(y_test, best.predict(x_test)))
ConfusionMatrixDisplay.from_estimator(best, x_test, y_test)
plt.show()